# 14 — Ghép head OS (05b) vào `sdc_iden.onnx`: 12 chuỗi → make/type/model

Bản ONNX của đúng mô hình mà notebook 13 xuất ra TFLite: ba head của `sdc_iden` + head OS của một run
`Models/<ts>_os` (05b) + `cascade()` **OS → type → make → model**.

| | `sdc_iden.onnx` (09) | `sdc_iden_os.onnx` (notebook này) |
|---|---|---|
| input | `input` `string [N,12]` + `thresholds` (tuỳ chọn) | **giữ nguyên** |
| output | `output` `string [N,3]` | **giữ nguyên** — `make`, `type`, `model` sau cascade |
| head OS | không | luật L0/L1 + rừng OS + cascade của 05b |
| runtime | onnxruntime | onnxruntime — **không cần Flex** như bản TFLite |

**Ghép chứ không dựng lại.** File `sdc_iden.onnx` đang deploy được nạp nguyên vẹn; chỉ node `Concat` ra
`output` bị thay. Head OS đọc lại các tensor graph đã tính sẵn (12 cột, 31 cờ số, token PRL), cascade đọc chỉ
số nhãn cuối của từng head (`sdc/final/<head>`), rồi tra nhãn và nối thành `output` như cũ. Phần iden vì vậy
giống bản deploy tới từng node; ngưỡng iden vẫn ghi đè được qua input `thresholds`.

**Head OS trong ONNX:**

* encoder OS: `ColumnTransformer` của bundle OS qua skl2onnx (như 09 làm với encoder iden), `mac_is_random`
  suy từ `mac_oui` theo cùng quy ước với notebook 13;
* rừng OS: duyệt cây bằng `Gather`/`LessOrEqual`/`Where` trên mảng phẳng của notebook 12, xác suất là trung
  bình float32 — **không** dùng `TreeEnsembleClassifier`, vì cộng dồn trọng số `1/300` đã làm tròn có thể đẩy
  một cửa sổ đúng `0.6` (180/300 cây) xuống dưới ngưỡng;
* luật `HOST_OS` / VCI: ONNX không có op `strip`/`lower`/thay thế chuỗi, nên `re.search(p, s.strip().lower())`
  được viết lại thành một `RegexFullMatch` trên chuỗi thô: khoảng trắng hai đầu nuốt bằng lớp ký tự khoảng
  trắng của Python, chữ hoa/thường bằng `(?i)`;
* chữ ký PRL: dùng token `dhcp_prl` tách theo `,` và `dhcp_prl_len` mà graph đã có — đúng với PRL dạng chuẩn
  mà contract 09 đã bắt buộc (thập phân, cách bởi `,`, không số 0 đầu). Bản TFLite (13) xử lý cả PRL lệch chuẩn.

Chạy được trên kernel thường của project (cần `onnx`, `onnxruntime`, `skl2onnx`) — không cần TensorFlow.

In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
# Nạp định nghĩa dùng chung từ 00_common.ipynb (không train).
from pathlib import Path

_cwd = Path.cwd().resolve()
_PIPELINE = next((d for p in (_cwd, *_cwd.parents)
                  for d in (p, p / "pipeline", p / "Code" / "pipeline")
                  if (d / "00_common.ipynb").is_file()), None)
assert _PIPELINE is not None, f"Không thấy pipeline/00_common.ipynb quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_PIPELINE / "00_common.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import ast
import hashlib
import inspect
import itertools
import json
import re
import warnings
from datetime import datetime

import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from IPython.display import display
from onnx import TensorProto as TP
from onnx import compose, helper
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

warnings.filterwarnings("ignore", message=".*InconsistentVersion.*")
ort.set_default_logger_severity(3)
print("onnx", onnx.__version__, "| onnxruntime", ort.__version__)

Configured for 88-core CPU
Gốc dự án: D:\01.AI_Security\02.SDC


Đã nạp hàm SDC từ 00_common.ipynb

onnx

1.22.0

| onnxruntime

1.28.0

## 1. Chọn run, nạp định nghĩa dùng lại

Định nghĩa lấy theo tên (AST) từ notebook gốc: bộ dựng graph `GB` và các khối của **09**, mảng rừng phẳng của
**12**, luật OS + `os_decide` + `cascade` của **05b**.

In [3]:
def import_defs(notebook, names, namespace):
    """Chạy các định nghĩa top-level (def/class/gán biến) có tên trong `names` của một notebook."""
    nb = json.loads(Path(notebook).read_text(encoding="utf-8"))
    wanted = []
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:
            continue
        for node in tree.body:
            name = (node.name if isinstance(node, (ast.FunctionDef, ast.ClassDef)) else
                    node.targets[0].id if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name)
                    else None)
            if name in names:
                wanted.append(node)
    exec(compile(ast.Module(body=wanted, type_ignores=[]), str(notebook), "exec"), namespace)
    missing = sorted(set(names) - set(namespace))
    assert not missing, f"{Path(notebook).name} thiếu: {missing}"
    return namespace


NB09 = import_defs(_PIPELINE / "out_model" / "09_build_sdc_iden.ipynb", [
    "FORMAT", "ONNX_OPSET", "IR_VERSION", "ONNX_LOCALE", "MISSING", "UNKNOWN_LABEL", "NOT_PRESENT", "HEADS",
    "MODEL_HEAD", "INPUT_COLUMNS", "IN_FEATURES", "IN_THRESHOLDS", "OUT_LABELS", "ENCODER_STRINGS",
    "L0_COLUMNS", "SOURCE_FLAGS", "SOURCE_EVIDENCE", "PRL_SEP", "_MAC_RE", "normalize_mac", "mac_is_random",
    "mac_oui", "_clean_text", "add_l0_columns", "to_input_matrix", "prl_options", "derive_numeric",
    "matrix_frame", "GB", "relink", "str_present", "float_eq", "set_locale", "assert_locale", "sha256_list",
], {"re": re, "np": np, "pd": pd, "onnx": onnx, "helper": helper, "TP": TP, "hashlib": hashlib})
NB12 = import_defs(_PIPELINE / "out_model" / "12_export_tflite_os.ipynb",
                   ["float32_floor", "flatten_forest"], {"np": np})
NB05B = import_defs(_PIPELINE / "train_model" / "05b_train_os_head.ipynb", [
    "UNK", "OS_CLASSES", "OS_THRESHOLD", "NOT_PRESENT", "HOST_OS", "mac_is_random", "prl_set",
    "l0_os", "l1_os", "os_decide", "TYPE_OK", "GENERAL_PURPOSE_TYPES", "MAKE_OK", "cascade",
], {"re": re, "np": np, "MISSING": MISSING, "apply_encoder": apply_encoder})
HEADS, MODEL_HEAD, UNK = NB09["HEADS"], NB09["MODEL_HEAD"], NB09["UNKNOWN_LABEL"]
COLS = list(NB09["INPUT_COLUMNS"])
assert NB05B["UNK"] == UNK

IDEN_RUN = None                   # None = *_iden mới nhất
OS_RUN = "20261002_163153_os"     # run *_os của 05b

iden_dir = (MODELS / IDEN_RUN) if IDEN_RUN else sorted(p for p in MODELS.glob("*_iden")
                                                       if (p / "sdc_iden.onnx").is_file())[-1]
onnx_path = iden_dir / "sdc_iden.onnx"
base_model = onnx.load(str(onnx_path))
base_contract = json.loads({p.key: p.value for p in base_model.metadata_props}["sdc.contract"])
assert base_contract["format"] == NB09["FORMAT"] and base_contract["columns"] == COLS
LABELS = base_contract["labels"]

os_dir = MODELS / OS_RUN
os_bundle = joblib.load(os_dir / "model.joblib")
assert os_bundle["format"] == "sdc-os-v0", os_bundle["format"]
assert NB05B["OS_THRESHOLD"] == os_bundle["threshold"] and list(NB05B["OS_CLASSES"]) == list(os_bundle["classes"])
MIN_JACCARD = inspect.signature(NB05B["l1_os"]).parameters["min_jaccard"].default
OS_LABELS = list(os_bundle["classes"]) + [UNK]
OSI = {c: i for i, c in enumerate(OS_LABELS)}
out_dir = os_dir / "onnx_iden_os"
print(f"iden: {iden_dir.name} ({onnx_path.stat().st_size / 1024:.0f} KiB)  |  OS: {OS_RUN}  |  lớp OS {OS_LABELS}")
print("đầu ra:", out_dir / "sdc_iden_os.onnx")

iden: 20261001_170502_iden (1598 KiB)  |  OS: 20261002_163153_os  |  lớp OS ['iOS', 'Android', 'Windows', 'macOS', 'Linux', 'Embedded', '__unknown__']

đầu ra:

D:\01.AI_Security\02.SDC\Models\20261002_163153_os\onnx_iden_os\sdc_iden_os.onnx

## 2. Tensor dùng lại từ graph đang deploy

Tên do `GB` của 09 sinh: `sdc/col/<cột>` (12 cột sau `Split`), `sdc/num/<cờ>` (`has_*`, `dhcp_opt_*`),
`sdc/n/prl_len`, `sdc/prl/tokens` (token `dhcp_prl` tách theo `,`), `sdc/final/<head>` (chỉ số nhãn sau
ngưỡng), bảng nhãn `sdc/lab/<head>`. Mỗi tên phải khớp **đúng một** tensor.

In [4]:
_outputs = [o for n in base_model.graph.node for o in n.output]
_inits = {i.name for i in base_model.graph.initializer}


def one(pattern, pool):
    hits = [o for o in pool if re.fullmatch(pattern, o)]
    assert len(hits) == 1, (pattern, hits)
    return hits[0]


split = next(n for n in base_model.graph.node if n.op_type == "Split" and n.input[0] == NB09["IN_FEATURES"])
COL = dict(zip(COLS, split.output))
IDEN_NUM = list(base_contract["derived_in_graph"]["columns"])
DERIVED = {c: one(rf"sdc/num/{re.escape(c)}_\d+", _outputs) for c in IDEN_NUM if c != "dhcp_prl_len"}
DERIVED["dhcp_prl_len"] = one(r"sdc/n/prl_len_\d+", _outputs)
PRL_TOKENS = one(r"sdc/prl/tokens_\d+", _outputs)
FINAL = {h: one(rf"sdc/final/{h}_\d+", _outputs) for h in HEADS}
LABEL_TABLE = {h: one(rf"sdc/lab/{h}_\d+", _inits) for h in HEADS}
OLD_OUT = [n for n in base_model.graph.node if NB09["OUT_LABELS"] in n.output]
assert len(OLD_OUT) == 1 and OLD_OUT[0].op_type == "Concat"
print(f"cột {len(COL)}, cờ số {len(DERIVED)}, token PRL {PRL_TOKENS}, chỉ số cuối {FINAL}")

cột 12, cờ số 31, token PRL sdc/prl/tokens_129, chỉ số cuối {'make': 'sdc/final/make_1083', 'type': 'sdc/final/type_1392', 'model': 'sdc/final/model_742'}

## 3. Dựng head OS + cascade và ghép

Mỗi khối theo đúng hàm tương ứng của 05b; mục 4 kiểm từng khối với chính hàm đó.

In [5]:
# --- Quy ước và hằng ----------------------------------------------------------------------
# Khoảng trắng mà str.strip() của Python bỏ: \t\n\v\f\r, \x1c-\x1f, \x85 và mọi ký tự nhóm Z.
WS = r"[\t\n\x{0b}\f\r\x{1c}-\x{1f}\x{85}\p{Z}]"
NOT_WS = "[^" + WS[1:]
OUI_BIT = "[0-9a-fA-F][2367abefABEF]:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}"
OUI_RE = re.compile(r"[0-9a-fA-F]{2}:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}")


def mac_random_from_oui(value):
    """mac_is_random từ `mac_oui` — <missing> = MAC ngẫu nhiên hoá (cùng quy ước notebook 13)."""
    v = str(value)
    if v in NB09["NOT_PRESENT"]:
        return 1
    return int(bool(OUI_RE.fullmatch(v)) and int(v[:2], 16) & 0x02 != 0)


def host_regex(pattern):
    """`re.search(pattern, raw.strip().lower())` -> một RegexFullMatch (RE2) trên chuỗi thô."""
    assert pattern.startswith("^") and not pattern.endswith(r"\$"), pattern
    q = pattern[1:]
    if q.endswith("$"):
        assert r"\b" not in q, pattern
        return f"(?s){WS}*(?i:{q[:-1]}){WS}*"
    if r"\b" in q:           # \b của Python là Unicode; RE2 thì không -> biên phải viết tường minh
        assert q.endswith(r"\b") and q.count(r"\b") == 1, pattern
        q = q[:-2] + r"(?:[^\p{L}\p{N}_]|$)"
    return f"(?s){WS}*(?i:{q}).*"


HOST_EXCLUDED = f"(?s){WS}*(?i:<missing>|nan|none)?{WS}*"          # h rỗng / <missing> / nan / none
VCI_MSFT = f"(?s){WS}*(?i:msft) .*{NOT_WS}.*"                       # strip().lower().startswith("msft ")
VCI_ANDROID = f"(?s){WS}*(?i:android-dhcp-).*"
MDNS_APPLE = "(?s).*(?:_companion link|_rdlink).*"

FLAT_OS = NB12["flatten_forest"](os_bundle["model"])
OS_WIDTH = int(os_bundle["model"].n_features_in_)
FOREST_OS = np.int64([OSI[str(c)] for c in os_bundle["model"].classes_])
SIG_KEYS = list(os_bundle["signatures"])
SIG_OPTIONS = sorted({o for k in SIG_KEYS for o in k})
SIG_MEMBER = np.float32([[o in k for k in SIG_KEYS] for o in SIG_OPTIONS])                       # [U, K]
SIG_SIZE = np.float32([[len(k) for k in SIG_KEYS]])                                               # [1, K]
SIG_OWNER = np.float32([[[c in os_bundle["signatures"][k] for c in OS_LABELS] for k in SIG_KEYS]])  # [1, K, O]
TYPE_L, MAKE_L, MODEL_L = LABELS["type"], LABELS["make"], LABELS[MODEL_HEAD]


# --- Khối graph ---------------------------------------------------------------------------
def I(gb, v, stem):
    return gb.const(np.int64([[v]]), stem)


def F(gb, v, stem):
    return gb.const(np.float32([[v]]), stem)


def rx(gb, tensor, pattern, stem):
    return gb.add("RegexFullMatch", [tensor], stem, pattern=pattern)


def build_random(gb, oui):
    """bool [N,1]: MAC ngẫu nhiên hoá."""
    absent = gb.add("Not", [NB09["str_present"](gb, oui, "oui")], "oui/absent")
    return gb.add("Or", [absent, rx(gb, oui, OUI_BIT, "oui/bit")], "random")


def build_os_encoder(gb, source):
    """ColumnTransformer của bundle OS qua skl2onnx -> (nodes, inits, opsets, tensor feature [N, 187])."""
    enc = os_bundle["encoder"]
    initial = ([(c, FloatTensorType([None, 1])) for c in enc["num_cols"]]
               + [(c, StringTensorType([None, 1])) for c in enc["cat_cols"] + enc["text_cols"]])
    ct = convert_sklearn(enc["ct"], "sdc_os_encoder", initial_types=initial, target_opset=NB09["ONNX_OPSET"])
    ct = compose.add_prefix(ct, "osenc/", rename_inputs=False)
    cols = enc["num_cols"] + enc["cat_cols"] + enc["text_cols"]
    nodes, inits = NB09["relink"](ct.graph, {i.name: source[c] for i, c in zip(ct.graph.input, cols)})
    return nodes, inits, list(ct.opset_import), ct.graph.output[0].name


def build_forest(gb, x, flat, width, stem):
    """Duyệt rừng phẳng (notebook 12): mỗi vòng đưa mọi cây xuống một tầng; xác suất = trung bình float32."""
    n = gb.add("Gather", [gb.add("Shape", [x], f"{stem}/shape"), gb.const(np.int64(0), f"{stem}/i0")],
               f"{stem}/n", axis=0)
    rows = gb.add("Range", [gb.const(np.int64(0), f"{stem}/r0"), n, gb.const(np.int64(1), f"{stem}/r1")],
                  f"{stem}/range")
    row = gb.add("Mul", [gb.add("Unsqueeze", [rows, gb.const(np.int64([1]), f"{stem}/ax1")], f"{stem}/rowu"),
                         gb.const(np.int64(width), f"{stem}/w")], f"{stem}/row")                  # [N, 1]
    shape = gb.add("Concat", [gb.add("Unsqueeze", [n, gb.const(np.int64([0]), f"{stem}/ax0")], f"{stem}/n1"),
                              gb.const(np.int64([len(flat["roots"])]), f"{stem}/t")], f"{stem}/nt", axis=0)
    node = gb.add("Expand", [gb.const(flat["roots"].astype(np.int64).reshape(1, -1), f"{stem}/roots"), shape],
                  f"{stem}/node")                                                                  # [N, T]
    xf = gb.add("Reshape", [x, gb.const(np.int64([-1]), f"{stem}/flat")], f"{stem}/xf")
    feature = gb.const(flat["feature"].astype(np.int64), f"{stem}/feature")
    threshold = gb.const(flat["threshold"], f"{stem}/threshold")
    left = gb.const(flat["left"].astype(np.int64), f"{stem}/left")
    right = gb.const(flat["right"].astype(np.int64), f"{stem}/right")
    for d in range(int(flat["depth"])):
        idx = gb.add("Add", [gb.add("Gather", [feature, node], f"{stem}/f", axis=0), row], f"{stem}/idx")
        xv = gb.add("Gather", [xf, idx], f"{stem}/xv", axis=0)
        le = gb.add("LessOrEqual", [xv, gb.add("Gather", [threshold, node], f"{stem}/thr", axis=0)], f"{stem}/le")
        node = gb.add("Where", [le, gb.add("Gather", [left, node], f"{stem}/l", axis=0),
                                gb.add("Gather", [right, node], f"{stem}/r", axis=0)], f"{stem}/node")
    values = gb.add("Gather", [gb.const(flat["value"], f"{stem}/value"), node], f"{stem}/values", axis=0)
    return gb.add("ReduceMean", [values, gb.const(np.int64([1]), f"{stem}/axt")], f"{stem}/proba", keepdims=0)


def build_os_head(gb, col, derived, prl_tokens, x_os, random):
    """`os_decide()` của 05b: L0 hostname -> L0 VCI -> L1 PRL -> L1 mDNS -> rừng. int64 [N,1]."""
    minus = I(gb, -1, "os/m1")
    host = minus
    for pattern, os_ in reversed(NB05B["HOST_OS"]):                       # luật đầu tiên khớp thắng
        host = gb.add("Where", [rx(gb, col["dhcp_hostname"], host_regex(pattern), "host/rx"),
                                I(gb, OSI[os_], "host/os"), host], "host")
    host = gb.add("Where", [rx(gb, col["dhcp_hostname"], HOST_EXCLUDED, "host/excl"), minus, host], "host/ok")

    android = gb.add("And", [rx(gb, col["dhcp_vci"], VCI_ANDROID, "vci/and"), random], "vci/and_rnd")
    vci = gb.add("Where", [rx(gb, col["dhcp_vci"], VCI_MSFT, "vci/msft"), I(gb, OSI["Windows"], "vci/win"),
                           gb.add("Where", [android, I(gb, OSI["Android"], "vci/android"), minus], "vci/a")], "vci")

    has = []
    for o in SIG_OPTIONS:
        eq = gb.add("Equal", [prl_tokens, gb.const(np.array([[str(o)]]), f"prl/o{o}", TP.STRING)], f"prl/eq{o}")
        cnt = gb.add("ReduceSum", [gb.add("Cast", [eq], f"prl/f{o}", to=TP.FLOAT),
                                   gb.const(np.int64([1]), f"prl/ax{o}")], f"prl/n{o}", keepdims=1)
        has.append(gb.add("Cast", [gb.add("Greater", [cnt, F(gb, 0.0, f"prl/z{o}")], f"prl/has{o}")],
                          f"prl/hf{o}", to=TP.FLOAT))
    has = gb.add("Concat", has, "prl/has", axis=1)                                             # [N, U]
    inter = gb.add("MatMul", [has, gb.const(SIG_MEMBER, "prl/member")], "prl/inter")            # [N, K]
    n_opt = derived["dhcp_prl_len"]                                                             # [N, 1]
    union = gb.add("Sub", [gb.add("Add", [n_opt, gb.const(SIG_SIZE, "prl/size")], "prl/u0"), inter], "prl/union")
    ratio = gb.add("Div", [inter, union], "prl/ratio")
    best = gb.add("ReduceMax", [ratio, gb.const(np.int64([1]), "prl/axk")], "prl/best", keepdims=1)
    tied = gb.add("Cast", [gb.add("Equal", [ratio, best], "prl/tie")], "prl/tied", to=TP.FLOAT)
    spread = gb.add("Mul", [gb.add("Unsqueeze", [tied, gb.const(np.int64([2]), "prl/ax2")], "prl/tu"),
                            gb.const(SIG_OWNER, "prl/owner")], "prl/spread")                    # [N, K, O]
    owners = gb.add("ReduceMax", [spread, gb.const(np.int64([1]), "prl/axs")], "prl/owners", keepdims=0)
    n_owner = gb.add("ReduceSum", [owners, gb.const(np.int64([1]), "prl/axo")], "prl/nown", keepdims=1)
    prl_os = gb.add("ArgMax", [owners], "prl/os", axis=1, keepdims=1)
    ok = gb.add("And", [gb.add("Greater", [n_opt, F(gb, 0.0, "prl/z")], "prl/any"),
                        gb.add("GreaterOrEqual", [best, F(gb, MIN_JACCARD, "prl/minj")], "prl/j")], "prl/ok1")
    ok = gb.add("And", [ok, gb.add("Equal", [n_owner, F(gb, 1.0, "prl/one")], "prl/uniq")], "prl/ok2")
    emb_rnd = gb.add("And", [gb.add("Equal", [prl_os, I(gb, OSI["Embedded"], "prl/emb")], "prl/isemb"), random],
                     "prl/embrnd")
    prl_ok = gb.add("And", [ok, gb.add("Not", [emb_rnd], "prl/notemb")], "prl/ok")

    mdns_ok = gb.add("And", [random, rx(gb, col["mdns_tokens"], MDNS_APPLE, "mdns/rx")], "mdns/ok")

    p = build_forest(gb, x_os, FLAT_OS, OS_WIDTH, "rf_os")
    pmax = gb.add("ReduceMax", [p, gb.const(np.int64([1]), "rf/ax")], "rf/pmax", keepdims=1)
    k = gb.add("ArgMax", [p], "rf/k", axis=1, keepdims=1)                 # chỉ số đầu khi hoà, như numpy
    forest_os = gb.add("Gather", [gb.const(FOREST_OS, "rf/map"), k], "rf/os", axis=0)
    tls_only = gb.add("Not", [gb.add("Or", [NB09["float_eq"](gb, derived["has_dhcp"], 1.0, "rf/dhcp"),
                                            NB09["float_eq"](gb, derived["has_mdns"], 1.0, "rf/mdns")], "rf/src")],
                      "rf/tls_only")
    general = gb.add("And", [gb.add("Not", [gb.add("Equal", [forest_os, I(gb, OSI["Embedded"], "rf/emb")], "rf/isemb")],
                                    "rf/notemb"), tls_only], "rf/general_tls")
    forest_ok = gb.add("And", [gb.add("GreaterOrEqual", [pmax, F(gb, float(os_bundle["threshold"]), "rf/thr")], "rf/conf"),
                               gb.add("Not", [general], "rf/gate")], "rf/ok")

    os_idx = gb.add("Where", [forest_ok, forest_os, I(gb, OSI[UNK], "os/unk")], "os/l2")
    os_idx = gb.add("Where", [mdns_ok, I(gb, OSI["iOS"], "os/ios"), os_idx], "os/mdns")
    os_idx = gb.add("Where", [prl_ok, prl_os, os_idx], "os/prl")
    os_idx = gb.add("Where", [gb.add("GreaterOrEqual", [vci, I(gb, 0, "os/z1")], "os/hv"), vci, os_idx], "os/vci")
    return gb.add("Where", [gb.add("GreaterOrEqual", [host, I(gb, 0, "os/z0")], "os/hh"), host, os_idx], "os")


def build_cascade(gb, os_idx, random, make, type_, model):
    """`cascade()` của 05b trên chỉ số nhãn int64 [N,1]."""
    eq = lambda t, v, s: gb.add("Equal", [t, I(gb, v, s)], s + "/eq")
    is_os = lambda *names: (eq(os_idx, OSI[names[0]], f"cas/{names[0]}") if len(names) == 1 else
                            gb.add("Or", [eq(os_idx, OSI[n], f"cas/{n}") for n in names], "cas/or"))
    lookup = lambda table, idx, s: gb.add("Gather", [gb.const(np.asarray(table), s), idx], s + "/g", axis=0)
    pair = lambda table, idx, width, s: lookup(np.asarray(table).ravel(), gb.add(
        "Add", [gb.add("Mul", [os_idx, I(gb, width, s + "/w")], s + "/m"), idx], s + "/i"), s)
    t_unk, m_unk, mo_unk = TYPE_L.index(UNK), MAKE_L.index(UNK), MODEL_L.index(UNK)
    known = gb.add("Not", [eq(os_idx, OSI[UNK], "cas/unk")], "cas/known")

    emb_general = gb.add("And", [is_os("Embedded"), lookup([t in NB05B["GENERAL_PURPOSE_TYPES"] for t in TYPE_L],
                                                           type_, "cas/general")], "cas/emb_general")
    type_ok = pair([[t in NB05B["TYPE_OK"].get(o, ()) for t in TYPE_L] for o in OS_LABELS], type_, len(TYPE_L),
                   "cas/type_ok")
    conflict = gb.add("And", [gb.add("And", [gb.add("Not", [eq(type_, t_unk, "cas/tu")], "cas/tk"),
                                             lookup([o in NB05B["TYPE_OK"] for o in OS_LABELS], os_idx, "cas/trule")],
                                     "cas/c1"), gb.add("Not", [type_ok], "cas/tbad")], "cas/conflict")
    drop_t = gb.add("And", [known, gb.add("Or", [emb_general, conflict], "cas/c2")], "cas/drop_t")
    type_ = gb.add("Where", [drop_t, I(gb, t_unk, "cas/t_unk"), type_], "cas/type1")
    empty = gb.add("And", [known, eq(type_, t_unk, "cas/te")], "cas/empty")
    phone = gb.add("And", [empty, gb.add("Or", [is_os("iOS"), gb.add("And", [is_os("Android"), random], "cas/ar")],
                                         "cas/ph")], "cas/phone")
    laptop = gb.add("And", [gb.add("And", [empty, gb.add("Not", [phone], "cas/np")], "cas/e2"),
                            is_os("Windows", "macOS")], "cas/laptop")
    type_ = gb.add("Where", [phone, I(gb, TYPE_L.index("Smartphone"), "cas/smart"),
                             gb.add("Where", [laptop, I(gb, TYPE_L.index("Laptop"), "cas/lap"), type_], "cas/t2")],
                   "cas/type")

    make_rule = lookup([o in NB05B["MAKE_OK"] for o in OS_LABELS], os_idx, "cas/mrule")
    make_ok = pair([[m in NB05B["MAKE_OK"].get(o, ()) for m in MAKE_L] for o in OS_LABELS], make, len(MAKE_L),
                   "cas/make_ok")
    bad = gb.add("And", [gb.add("And", [gb.add("Not", [eq(make, m_unk, "cas/mu")], "cas/mk"), make_rule], "cas/b1"),
                         gb.add("Not", [make_ok], "cas/mbad")], "cas/bad")
    make = gb.add("Where", [gb.add("And", [known, bad], "cas/kb"), I(gb, m_unk, "cas/m_unk"), make], "cas/make1")
    apple = MAKE_L.index("Apple")
    stray = gb.add("And", [gb.add("And", [known, eq(make, apple, "cas/ap")], "cas/ka"),
                           gb.add("Not", [is_os("iOS", "macOS")], "cas/nonapple")], "cas/stray")
    make = gb.add("Where", [stray, I(gb, m_unk, "cas/m_unk2"), make], "cas/make2")
    fill = gb.add("And", [gb.add("And", [known, eq(make, m_unk, "cas/me")], "cas/f1"), make_rule], "cas/fill")
    make = gb.add("Where", [fill, I(gb, apple, "cas/apple"), make], "cas/make")

    gone = gb.add("Or", [eq(make, m_unk, "cas/mu2"), eq(type_, t_unk, "cas/tu2")], "cas/gone")
    drop_m = gb.add("And", [gone, gb.add("Not", [eq(model, mo_unk, "cas/mou")], "cas/mok")], "cas/drop_m")
    return make, type_, gb.add("Where", [drop_m, I(gb, mo_unk, "cas/mo_unk"), model], "cas/model")


# --- Sắp xếp topo + bỏ node chết -------------------------------------------------------------
def topo_prune(nodes, inits, inputs, outputs):
    """Sắp xếp topo (checker bắt buộc), rồi bỏ node/initializer không dẫn tới output."""
    ready, order, pending = set(inputs) | {i.name for i in inits}, [], list(nodes)
    while pending:
        rest = []
        for n in pending:
            if all((not i) or i in ready for i in n.input):
                order.append(n)
                ready.update(n.output)
            else:
                rest.append(n)
        assert len(rest) < len(pending), "graph có vòng hoặc thiếu tensor"
        pending = rest
    need, keep = set(outputs), []
    for n in reversed(order):
        if any(o in need for o in n.output):
            keep.append(n)
            need.update(i for i in n.input if i)
    keep.reverse()
    used = {i for n in keep for i in n.input}
    return keep, [t for t in inits if t.name in used or t.name in inputs]


# --- Lắp ráp -----------------------------------------------------------------------------
gb = NB09["GB"]("sdcos")
random = build_random(gb, COL["mac_oui"])
source = {**COL, **DERIVED, "mac_is_random": gb.add("Cast", [random], "random_f", to=TP.FLOAT)}
enc_nodes, enc_inits, enc_opsets, x_os = build_os_encoder(gb, source)
os_idx = build_os_head(gb, COL, DERIVED, PRL_TOKENS, x_os, random)
make, type_, model_idx = build_cascade(gb, os_idx, random, FINAL["make"], FINAL["type"], FINAL[MODEL_HEAD])
labels = [gb.add("Gather", [LABEL_TABLE[h], t], f"out/{h}", axis=0)
          for h, t in zip(HEADS, (make, type_, model_idx))]                      # mỗi head string [N,1]
gb.nodes.append(helper.make_node("Concat", labels, [NB09["OUT_LABELS"]], name=gb.name("n_out"), axis=1))

g = base_model.graph
graph_inputs = [i.name for i in g.input]
nodes, inits = topo_prune([n for n in g.node if n is not OLD_OUT[0]] + enc_nodes + gb.nodes,
                          list(g.initializer) + enc_inits + gb.inits, graph_inputs, [NB09["OUT_LABELS"]])
opsets = {o.domain: o.version for o in base_model.opset_import}
for o in enc_opsets:
    opsets[o.domain] = max(opsets.get(o.domain, 0), o.version)
graph = helper.make_graph(nodes, "sdc_iden_os", list(g.input), list(g.output), inits)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid(d, v) for d, v in opsets.items()])
model.ir_version = NB09["IR_VERSION"]
patched = NB09["set_locale"](model)
NB09["assert_locale"](model)
onnx.checker.check_model(model)
print(f"node {len(g.node)} -> {len(nodes)}  |  StringNormalizer đặt locale C: {len(patched)}  |  opset {opsets}")

node 633 -> 1080  |  StringNormalizer đặt locale C: 6  |  opset {'': 20, 'ai.onnx.ml': 3, 'com.microsoft': 1}

## 4. Kiểm nhanh

Hai phép kiểm, vài giây, đều so với **chính code Python của 05b**:

1. **Cascade** — dựng riêng khối cascade thành một model nhỏ, chạy mọi tổ hợp (OS, MAC ngẫu nhiên, make, type,
   model), so với `cascade()`.
2. **Đầu cuối** — bộ capture và các dòng ép từng tầng OS (mọi luật hostname kể cả hoa/thường, khoảng trắng,
   chữ Việt sát biên `\b`; VCI; PRL quanh từng chữ ký ở dạng chuẩn; mDNS Apple; MAC thiếu / thật / ngẫu nhiên):
   output của file mới so với `cascade(os_decide(...), nhãn của sdc_iden.onnx gốc)`.

In [6]:
# 1. Cascade: mọi tổ hợp
cg = NB09["GB"]("cas")
cin = [helper.make_tensor_value_info(n, TP.INT64 if n != "random" else TP.BOOL, [None, 1])
       for n in ("os", "random", "make", "type", "model")]
outs = build_cascade(cg, "os", "random", "make", "type", "model")
cmodel = helper.make_model(helper.make_graph(
    cg.nodes + [helper.make_node("Concat", list(outs), ["out"], axis=1)], "cascade", cin,
    [helper.make_tensor_value_info("out", TP.INT64, [None, 3])], cg.inits),
    opset_imports=[helper.make_opsetid("", NB09["ONNX_OPSET"])])
cmodel.ir_version = NB09["IR_VERSION"]
combos = np.array(list(itertools.product(range(len(OS_LABELS)), (0, 1), range(len(MAKE_L)), range(len(TYPE_L)),
                                         range(len(MODEL_L)))), dtype=np.int64)
got = ort.InferenceSession(cmodel.SerializeToString(), providers=["CPUExecutionProvider"]).run(None, {
    "os": combos[:, [0]], "random": combos[:, [1]].astype(bool), "make": combos[:, [2]],
    "type": combos[:, [3]], "model": combos[:, [4]]})[0]
bad = sum(NB05B["cascade"](OS_LABELS[o], r, MAKE_L[mk], TYPE_L[ty], MODEL_L[mo])[1:4]
          != (MAKE_L[g_[0]], TYPE_L[g_[1]], MODEL_L[g_[2]]) for (o, r, mk, ty, mo), g_ in zip(combos, got))
print(f"cascade: {len(combos)} tổ hợp, lệch {bad}")
assert bad == 0

cascade: 7560 tổ hợp, lệch 0

In [7]:
# 2. Đầu cuối
def capture_frame():
    feat = DATA / "features"
    sess = pd.read_csv(feat / "session_features_capture.csv")
    dhcp = pd.read_csv(feat / "dhcp_features_capture.csv")
    hostnames = dhcp.dropna(subset=["hostname"]).drop_duplicates("mac").set_index("mac")["hostname"]
    sess["dhcp_hostname"] = sess["mac"].map(hostnames).fillna(MISSING)
    sess["mdns_model"] = MISSING
    return NB09["add_l0_columns"](sess)


def os_probe(base):
    hostnames = ["iPhone", " iPhone ", "IPHONE-cua-Lan", "iphoneđ", "iphone-đ", "ipad-pro", "ipod", "MacBook-Air",
                 "imac", "mac-mini", "macpro", "android-0123456789abcdef", "Galaxy-A51", "sm-a515f", "redmi-9",
                 "xiaomi", "poco-x3", "mi-9", "realme-6i", "oppo-a92", "vivo-1906", "oneplus", "pixel-7",
                 "huawei-p30", "honor-x8", "nokia", "moto-g", "DESKTOP-AB12CD3", "desktop-ducanh", " Desktop-AB12 ",
                 "laptop-vaf70sq6", "win-abcd1234", "WIN-AB", "raspberrypi", "raspberrypi4", "ubuntu", "debian-x",
                 "fedora", "archlinux", "ESP_0C3994", "tuya-plug", "shelly1", "tasmota", "wled-1", "linova",
                 "Johns-iPhone", "ĐIỆN-THOẠI", "none", " NaN ", "", MISSING]
    vcis = ["MSFT 5.0", " msft 5.0", "MSFT5.0", "msft ", "MSFT \t", "android-dhcp-11", "ANDROID-DHCP-16",
            "android-dhcp", "dhcpcd-6.8.2:Linux-4.4.22+:armv7l", "udhcp 1.26.2", MISSING]
    prls = []
    for key in os_bundle["signatures"]:
        opts = [str(v) for v in sorted(key)]
        prls += [",".join(opts), ",".join(reversed(opts)), ",".join(opts[:-1]), ",".join(opts + ["162"]),
                 ",".join([opts[0]] + opts)]
    mdns = ["_companion link _tcp local", "_rdlink _tcp local", "x _companion link", "_airplay _tcp local", MISSING]
    ouis = [MISSING, "00:e0:4c", "02:e0:4c", "00:11:22", "AA:BB:CC", "fe:db:aa"]   # thiếu, thật, ngẫu nhiên...
    rows = []
    for b, o in itertools.product(base, ouis):
        rows += [dict(b, dhcp_hostname=h, mac_oui=o) for h in hostnames]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=v, mac_oui=o) for v in vcis]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=p, mac_oui=o) for p in prls]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=MISSING, mdns_tokens=m, mac_oui=o)
                 for m in mdns]
    return np.asarray([[str(r[c]) for c in COLS] for r in rows], dtype=object)


def reference(m):
    iden = base_sess.run([NB09["OUT_LABELS"]], {NB09["IN_FEATURES"]: m})[0].astype(str)
    frame = NB09["matrix_frame"](m, COLS, IDEN_NUM)
    frame["mac_is_random"] = [mac_random_from_oui(v) for v in frame["mac_oui"]]
    decided = NB05B["os_decide"](os_bundle, frame)
    out = [NB05B["cascade"](d[0], r, *row)[1:4] for d, r, row in zip(decided, frame["mac_is_random"], iden)]
    return np.asarray(out, dtype=str).reshape(-1, 3), iden


base_sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
new_sess = ort.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"])
capture = capture_frame()
cap_m = NB09["to_input_matrix"](capture, COLS)
checks = {"capture": cap_m, "ép OS": os_probe(capture.drop_duplicates("mac").head(5)[COLS].to_dict("records"))}
rows = []
for name, m in checks.items():
    want, iden = reference(m)
    got = new_sess.run([NB09["OUT_LABELS"]], {NB09["IN_FEATURES"]: m})[0].astype(str)
    rows.append({"nguồn": name, "dòng": len(m), **{f"{h} lệch": int((want[:, i] != got[:, i]).sum())
                                                   for i, h in enumerate(HEADS)},
                 "ô đổi bởi cascade": int((want != iden).sum())})
check = pd.DataFrame(rows).set_index("nguồn")
display(check)
assert check.filter(like=" lệch").to_numpy().sum() == 0
print("OK — file mới trả đúng cascade(os_decide(), sdc_iden.onnx) của 05b")

sample = pd.DataFrame(new_sess.run([NB09["OUT_LABELS"]], {NB09["IN_FEATURES"]: cap_m})[0], columns=HEADS)
sample.insert(0, "dhcp_hostname", cap_m[:, COLS.index("dhcp_hostname")])
sample.insert(1, "iden gốc", ["/".join(r) for r in reference(cap_m)[1]])
display(sample.drop_duplicates().head(20))

,dòng,make lệch,type lệch,model lệch,ô đổi bởi cascade
nguồn,,,,,
capture,62,0,0,0,3
ép OS,5010,0,0,0,3215


OK — file mới trả đúng cascade(os_decide(), sdc_iden.onnx) của 05b

,dhcp_hostname,iden gốc,make,type,model
0,lee_kingdom,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
1,iphone,Apple/Smartphone/iPhone,Apple,Smartphone,iPhone
2,linova,Linova/Linux/Laptop/Linova Laptop HP,Linova/Linux,Laptop,Linova Laptop HP
3,desktop-ducanh,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
4,<missing>,Generic Laptop/Laptop/Windows Laptop HP,Generic Laptop,Laptop,Windows Laptop HP
5,desktop-dngjhrt,Generic Laptop/Laptop/Windows Desktop HP,Generic Laptop,Laptop,Windows Desktop HP
6,<missing>,Camera/IP Camera/Generic IP Camera,Camera,IP Camera,Generic IP Camera
7,<missing>,Xiaomi/Smartphone/Redmi Note 10,Xiaomi,Smartphone,Redmi Note 10
8,samsung-cua-duc-anh,Samsung/Smartphone/Samsung Galaxy,Samsung,Smartphone,Samsung Galaxy
9,raspberrypi,Raspberry Pi/Single-board Computer/Raspberry Pi,Raspberry Pi,Single-board Computer,Raspberry Pi


## 5. Lưu

Contract của 09 giữ nguyên các khoá chung (cột, nhãn, hash, hierarchy, ngưỡng iden), thêm `os_head` (luật,
chữ ký, cascade, quy ước `mac_is_random`) và nguồn. Thư mục `Models/<OS_RUN>/onnx_iden_os/` — tên không khớp
`*_iden`, nên 05b/13 vẫn lấy đúng `sdc_iden.onnx` gốc làm "iden mới nhất".

In [8]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_iden_os")
contract = {k: v for k, v in base_contract.items() if k not in ("size_optimization",)}
contract.update({
    "format": "sdc-iden-os-onnx-v1", "run_id": run_id, "created": datetime.now().isoformat(timespec="seconds"),
    "source_iden_onnx": {"run_id": iden_dir.name, "sha256": hashlib.sha256(onnx_path.read_bytes()).hexdigest()},
    "output_note": "make/type/model sau cascade OS -> type -> make -> model (05b)",
    "os_head": {
        "run_id": OS_RUN, "classes": OS_LABELS, "forest_threshold": float(os_bundle["threshold"]),
        "mac_is_random": "từ cột mac_oui: <missing>/rỗng = MAC ngẫu nhiên hoá (router luôn biết MAC, extractor "
                         "chỉ trả <missing> khi MAC ngẫu nhiên); OUI gửi nguyên thì xét bit U/L 0x02 của octet đầu",
        "l0_hostname": [{"search": p, "os": o} for p, o in NB05B["HOST_OS"]],
        "l0_vci": [{"prefix": "msft ", "os": "Windows"},
                   {"prefix": "android-dhcp-", "os": "Android", "require_mac_random": True}],
        "l1_prl": {"min_jaccard": MIN_JACCARD, "prl_format": "dạng chuẩn của contract (như 09)",
                   "signatures": [{"options": sorted(int(v) for v in k), "os": sorted(o)}
                                  for k, o in os_bundle["signatures"].items()]},
        "l1_mdns": {"contains_any": ["_companion link", "_rdlink"], "os": "iOS", "require_mac_random": True},
        "forest_gate": "lớp khác Embedded thì cửa sổ phải có DHCP hoặc mDNS",
        "cascade": {"type_ok": {k: sorted(v) for k, v in NB05B["TYPE_OK"].items()},
                    "general_purpose_types": sorted(NB05B["GENERAL_PURPOSE_TYPES"]),
                    "make_ok": {k: sorted(v) for k, v in NB05B["MAKE_OK"].items()},
                    "fill": {"iOS": "Smartphone + Apple", "Android + MAC ngẫu nhiên": "Smartphone",
                             "Windows": "Laptop", "macOS": "Laptop + Apple"},
                    "model": "make hoặc type là __unknown__ thì model cũng __unknown__"},
    },
})
del model.metadata_props[:]
for key, value in (("sdc.contract", json.dumps(contract, ensure_ascii=False)),
                   ("sdc.columns_sha256", contract["columns_sha256"]),
                   ("sdc.labels_sha256", contract["labels_sha256"]),
                   ("sdc.run_id", run_id), ("sdc.format", contract["format"])):
    model.metadata_props.append(onnx.StringStringEntryProto(key=key, value=value))

out_dir.mkdir(parents=True, exist_ok=True)
payload = model.SerializeToString()
(out_dir / "sdc_iden_os.onnx").write_bytes(payload)
(out_dir / "sdc_iden_os_contract.json").write_text(json.dumps(contract, ensure_ascii=False, indent=1),
                                                   encoding="utf-8")
(out_dir / "meta.json").write_text(json.dumps({
    "format": contract["format"], "run_id": run_id, "created": contract["created"],
    "source_iden_onnx": contract["source_iden_onnx"], "os_run": OS_RUN, "bytes": len(payload),
    "check": check.reset_index().to_dict("records"), "cascade_combos_checked": int(len(combos)),
}, indent=2, ensure_ascii=False, default=int), encoding="utf-8")
for p in sorted(out_dir.iterdir()):
    print(f"{p.stat().st_size / 1024:9.1f} KiB  {p}")

      0.7 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\onnx_iden_os\meta.json

   2284.5 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\onnx_iden_os\sdc_iden_os.onnx

     12.6 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\onnx_iden_os\sdc_iden_os_contract.json